# CFEU journal extension: Notebook 3 of 3, explainability, fairness and robustness

Run after Notebooks 1 and 2. It reads their outputs from `cfeu_journal` in your Google Drive and takes about 20–30 minutes.

**What it tests**
1. **Explainability:** which features drive the combined model, whether the adverse-action reasons stay the same when the model is retrained with a different seed (stability), and whether fixing a declined applicant's top reason actually lowers their risk score more than changing a random feature (faithfulness).
2. **Fairness:** approval rates by gender and age band at a 70% approval level, for the bureau-only and combined models, overall and for thin-file applicants.
3. **Berka robustness:** whether the strong cash-flow result holds when the most powerful features are removed, when a 30-day gap is left before the loan date, and when only finished loans are used.

**Send back to Claude:** `explain_results.json` (downloads at the end; summary numbers only).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, time
import numpy as np, pandas as pd
OUT = '/content/drive/MyDrive/cfeu_journal'
!pip -q install lightgbm pyarrow

In [ ]:
# Shared functions (no need to edit)
# ---- Modelling and evaluation (Notebook 2) ----
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

try:
    import lightgbm as lgb
    HAVE_LGB = True
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAVE_LGB = False


def make_model(kind, seed, small_data=False):
    if kind == "logit":
        return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                             LogisticRegression(max_iter=2000, C=1.0))
    if HAVE_LGB:
        p = dict(n_estimators=150, learning_rate=0.05, num_leaves=7, min_child_samples=20,
                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8) if small_data else \
            dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0)
        return lgb.LGBMClassifier(random_state=seed, verbose=-1, n_jobs=-1, **p)
    return HistGradientBoostingClassifier(max_iter=150 if small_data else 400, learning_rate=0.05,
                                          max_leaf_nodes=7 if small_data else 31, random_state=seed)


def oof_predictions(X, y, kind, seeds, n_splits=5, small_data=False):
    """Out-of-fold predicted default probabilities, one column per seed."""
    P = np.zeros((len(y), len(seeds)))
    for j, s in enumerate(seeds):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=s)
        for tr, te in skf.split(X, y):
            m = make_model(kind, s, small_data)
            m.fit(X.iloc[tr], y[tr])
            P[te, j] = m.predict_proba(X.iloc[te])[:, 1]
    return P


def ks_stat(y, p):
    return float(stats.ks_2samp(p[y == 1], p[y == 0]).statistic)


def approval_metrics(y, p, approve_rate=0.7):
    """Approve the lowest-risk share; report default rate among approved."""
    cut = np.quantile(p, approve_rate)
    ap = p <= cut
    return {"approve_rate": approve_rate, "default_rate_approved": float(y[ap].mean()),
            "default_rate_declined": float(y[~ap].mean()), "threshold": float(cut)}


def calib_bins(y, p, n_bins=10):
    q = pd.qcut(p, n_bins, labels=False, duplicates="drop")
    d = pd.DataFrame({"y": y, "p": p, "bin": q}).groupby("bin").agg(pred=("p", "mean"), obs=("y", "mean"), n=("y", "size"))
    return d.reset_index().to_dict(orient="list")


def bootstrap_auc_ci(y, p, n_boot=500, seed=0):
    rng = np.random.default_rng(seed); n = len(y); vals = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if y[i].min() == y[i].max():
            continue
        vals.append(roc_auc_score(y[i], p[i]))
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]


# Fast DeLong test (Sun & Xu 2014) for two correlated AUCs on the same cases
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N); T2[J] = T
    return T2


def delong_test(y, p1, p2):
    y = np.asarray(y); order = np.argsort(-y, kind="mergesort")
    y = y[order]; preds = np.vstack([p1, p2])[:, order]
    m = int(y.sum()); n = len(y) - m
    pos, neg = preds[:, :m], preds[:, m:]
    tx = np.array([_midrank(r) for r in pos]); ty = np.array([_midrank(r) for r in neg])
    tz = np.array([_midrank(r) for r in preds])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    diff = aucs[0] - aucs[1]
    se = np.sqrt(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1])
    z = diff / se if se > 0 else 0.0
    return {"auc_1": float(aucs[0]), "auc_2": float(aucs[1]), "diff": float(diff),
            "z": float(z), "p_value": float(2 * stats.norm.sf(abs(z)))}


def evaluate(y, P, mask=None, n_boot=500):
    """P: n x seeds out-of-fold probabilities. Seed spread plus bootstrap CI on the seed-averaged score."""
    y = np.asarray(y)
    if mask is not None:
        y, P = y[mask], P[mask]
    per_seed = [roc_auc_score(y, P[:, j]) for j in range(P.shape[1])]
    p = P.mean(axis=1)
    return {"n": int(len(y)), "default_rate": float(y.mean()),
            "auc_mean_over_seeds": float(np.mean(per_seed)), "auc_sd_over_seeds": float(np.std(per_seed)),
            "auc": float(roc_auc_score(y, p)), "auc_ci95": bootstrap_auc_ci(y, p, n_boot),
            "pr_auc": float(average_precision_score(y, p)), "brier": float(brier_score_loss(y, p)),
            "ks": ks_stat(y, p), "approval_at_70pct": approval_metrics(y, p, 0.7),
            "calibration": calib_bins(y, p)}


def run_suite(df, y, feature_sets, seeds, subgroups, small_data=False, n_boot=500, log=print):
    """Train every learner on every feature set; evaluate overall and per subgroup; DeLong vs the first set."""
    y = np.asarray(y).astype(int)
    subgroups = {g: (None if m is None else np.asarray(m, bool)) for g, m in subgroups.items()}
    results, oof = {}, {}
    for kind in ["logit", "gbm"]:
        for name, cols in feature_sets.items():
            log(f"  {kind:5s} | {name:12s} | {len(cols)} features")
            P = oof_predictions(df[cols], y, kind, seeds, small_data=small_data)
            oof[(kind, name)] = P
            results[f"{kind}|{name}"] = {g: evaluate(y, P, m, n_boot) for g, m in subgroups.items()}
    base = list(feature_sets)[0]
    tests = {}
    for kind in ["logit", "gbm"]:
        for name in list(feature_sets)[1:]:
            for g, m in subgroups.items():
                mm = np.ones(len(y), bool) if m is None else m
                tests[f"{kind}|{name} vs {base}|{g}"] = delong_test(
                    y[mm], oof[(kind, name)][mm].mean(1), oof[(kind, base)][mm].mean(1))
    return results, tests, oof


def headline_table(results):
    rows = []
    for key, groups in results.items():
        kind, fs = key.split("|")
        for g, r in groups.items():
            rows.append({"learner": kind, "features": fs, "group": g, "n": r["n"],
                         "AUC": round(r["auc"], 3), "AUC 95% CI": f"{r['auc_ci95'][0]:.3f}-{r['auc_ci95'][1]:.3f}",
                         "AUC sd (seeds)": round(r["auc_sd_over_seeds"], 4), "PR-AUC": round(r["pr_auc"], 3),
                         "Brier": round(r["brier"], 4), "KS": round(r["ks"], 3),
                         "Default rate if 70% approved": round(r["approval_at_70pct"]["default_rate_approved"], 4)})
    return pd.DataFrame(rows)


# ---- Home Credit feature building ----
import os, json
import numpy as np
import pandas as pd


def _safe_div(a, b):
    a = np.asarray(a, dtype="float64"); b = np.asarray(b, dtype="float64")
    out = np.full_like(a, np.nan)
    m = (b != 0) & ~np.isnan(b)
    out[m] = a[m] / b[m]
    return out


def build_home_credit(hc_dir):
    """Return one row per applicant with TARGET, feature groups and audit-only fields."""
    app = pd.read_csv(os.path.join(hc_dir, "application_train.csv"), usecols=[
        "SK_ID_CURR", "TARGET", "CODE_GENDER", "DAYS_BIRTH", "DAYS_EMPLOYED",
        "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
        "NAME_CONTRACT_TYPE", "NAME_INCOME_TYPE",
        "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"])
    df = app[["SK_ID_CURR", "TARGET"]].copy()

    # Audit-only fields: never used as model inputs
    df["audit_gender"] = app["CODE_GENDER"].where(app["CODE_GENDER"].isin(["M", "F"]))
    age = -app["DAYS_BIRTH"] / 365.25
    df["audit_age"] = age.round(1)
    df["audit_age_band"] = pd.cut(age, [0, 30, 50, 200], labels=["<30", "30-49", "50+"], right=False).astype(str)
    df["contract_type"] = app["NAME_CONTRACT_TYPE"]

    # Bureau group: external scores + credit bureau records
    for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]:
        df["bur_" + c.lower()] = app[c]

    bur = pd.read_csv(os.path.join(hc_dir, "bureau.csv"), usecols=[
        "SK_ID_CURR", "SK_ID_BUREAU", "CREDIT_ACTIVE", "DAYS_CREDIT", "CREDIT_DAY_OVERDUE",
        "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE", "CNT_CREDIT_PROLONG"])
    bur["is_active"] = (bur["CREDIT_ACTIVE"] == "Active").astype(int)
    g = bur.groupby("SK_ID_CURR")
    b = pd.DataFrame({
        "bur_n_loans": g.size(),
        "bur_n_active": g["is_active"].sum(),
        "bur_oldest_years": -g["DAYS_CREDIT"].min() / 365.25,
        "bur_newest_years": -g["DAYS_CREDIT"].max() / 365.25,
        "bur_max_days_overdue": g["CREDIT_DAY_OVERDUE"].max(),
        "bur_total_credit": g["AMT_CREDIT_SUM"].sum(),
        "bur_total_debt": g["AMT_CREDIT_SUM_DEBT"].sum(),
        "bur_total_overdue": g["AMT_CREDIT_SUM_OVERDUE"].sum(),
        "bur_n_prolonged": g["CNT_CREDIT_PROLONG"].sum(),
    })
    b["bur_debt_to_credit"] = _safe_div(b["bur_total_debt"], b["bur_total_credit"])

    bb_path = os.path.join(hc_dir, "bureau_balance.csv")
    if os.path.exists(bb_path):
        bb = pd.read_csv(bb_path, usecols=["SK_ID_BUREAU", "STATUS"])
        bb["late"] = bb["STATUS"].astype(str).isin(list("12345")).astype(int)
        bb = bb.groupby("SK_ID_BUREAU")["late"].agg(["sum", "count"]).reset_index()
        bb = bb.merge(bur[["SK_ID_BUREAU", "SK_ID_CURR"]], on="SK_ID_BUREAU")
        bb = bb.groupby("SK_ID_CURR")[["sum", "count"]].sum()
        b["bur_late_month_share"] = _safe_div(bb["sum"].reindex(b.index), bb["count"].reindex(b.index))
    df = df.merge(b, left_on="SK_ID_CURR", right_index=True, how="left")
    df["thin_file"] = df["bur_n_loans"].isna().astype(int)
    df["bur_n_loans"] = df["bur_n_loans"].fillna(0)
    df["bur_n_active"] = df["bur_n_active"].fillna(0)

    # Alternative group: affordability from the application + repayment behaviour outside the bureau
    df["alt_income"] = app["AMT_INCOME_TOTAL"]
    df["alt_credit_to_income"] = _safe_div(app["AMT_CREDIT"], app["AMT_INCOME_TOTAL"])
    df["alt_annuity_to_income"] = _safe_div(app["AMT_ANNUITY"], app["AMT_INCOME_TOTAL"])
    df["alt_credit_to_goods"] = _safe_div(app["AMT_CREDIT"], app["AMT_GOODS_PRICE"])
    emp = app["DAYS_EMPLOYED"].where(app["DAYS_EMPLOYED"] != 365243)  # 365243 is a placeholder for not employed
    df["alt_employment_years"] = -emp / 365.25
    df["alt_not_employed"] = (app["DAYS_EMPLOYED"] == 365243).astype(int)
    df["alt_income_type"] = app["NAME_INCOME_TYPE"].astype("category").cat.codes

    prev = pd.read_csv(os.path.join(hc_dir, "previous_application.csv"),
                       usecols=["SK_ID_CURR", "NAME_CONTRACT_STATUS", "AMT_APPLICATION", "AMT_CREDIT"])
    prev["approved"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype(int)
    prev["refused"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
    gp = prev.groupby("SK_ID_CURR")
    p = pd.DataFrame({"alt_prev_n": gp.size(), "alt_prev_approved_share": gp["approved"].mean(),
                      "alt_prev_refused_share": gp["refused"].mean()})

    ins = pd.read_csv(os.path.join(hc_dir, "installments_payments.csv"), usecols=[
        "SK_ID_CURR", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"],
        dtype={"SK_ID_CURR": "int32", "DAYS_INSTALMENT": "float32", "DAYS_ENTRY_PAYMENT": "float32",
               "AMT_INSTALMENT": "float32", "AMT_PAYMENT": "float32"})
    ins["days_late"] = (ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]).clip(lower=0)
    ins["is_late"] = (ins["days_late"] > 0).astype("int8")
    ins["is_short"] = (ins["AMT_PAYMENT"] < ins["AMT_INSTALMENT"] * 0.99).astype("int8")
    gi = ins.groupby("SK_ID_CURR")
    i = pd.DataFrame({"alt_inst_n": gi.size(), "alt_inst_late_share": gi["is_late"].mean(),
                      "alt_inst_mean_days_late": gi["days_late"].mean(), "alt_inst_max_days_late": gi["days_late"].max(),
                      "alt_inst_short_share": gi["is_short"].mean(),
                      "alt_inst_paid_ratio": gi["AMT_PAYMENT"].sum() / gi["AMT_INSTALMENT"].sum().replace(0, np.nan)})
    del ins

    pos = pd.read_csv(os.path.join(hc_dir, "POS_CASH_balance.csv"), usecols=["SK_ID_CURR", "SK_DPD_DEF"])
    pos["dpd"] = (pos["SK_DPD_DEF"] > 0).astype("int8")
    gq = pos.groupby("SK_ID_CURR")
    q = pd.DataFrame({"alt_pos_dpd_share": gq["dpd"].mean(), "alt_pos_max_dpd": gq["SK_DPD_DEF"].max()})
    del pos

    cc = pd.read_csv(os.path.join(hc_dir, "credit_card_balance.csv"),
                     usecols=["SK_ID_CURR", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "SK_DPD_DEF"])
    cc["util"] = _safe_div(cc["AMT_BALANCE"], cc["AMT_CREDIT_LIMIT_ACTUAL"])
    cc["dpd"] = (cc["SK_DPD_DEF"] > 0).astype("int8")
    gc = cc.groupby("SK_ID_CURR")
    c = pd.DataFrame({"alt_cc_util_mean": gc["util"].mean(), "alt_cc_dpd_share": gc["dpd"].mean()})
    del cc

    for t in (p, i, q, c):
        df = df.merge(t, left_on="SK_ID_CURR", right_index=True, how="left")
    df["alt_prev_n"] = df["alt_prev_n"].fillna(0)
    df["alt_inst_n"] = df["alt_inst_n"].fillna(0)
    return df


def home_credit_summary(df):
    bur_cols = [c for c in df.columns if c.startswith("bur_")]
    alt_cols = [c for c in df.columns if c.startswith("alt_")]
    out = {"n_applicants": int(len(df)), "default_rate": round(float(df["TARGET"].mean()), 4),
           "n_bureau_features": len(bur_cols), "n_alternative_features": len(alt_cols), "groups": {}}
    for name, sub in [("thin_file", df[df.thin_file == 1]), ("with_bureau", df[df.thin_file == 0])]:
        out["groups"][name] = {
            "n": int(len(sub)), "share": round(len(sub) / len(df), 4),
            "default_rate": round(float(sub["TARGET"].mean()), 4),
            "ext_source_missing_share": {c: round(float(sub[c].isna().mean()), 4)
                                         for c in ["bur_ext_source_1", "bur_ext_source_2", "bur_ext_source_3"]},
            "has_previous_loans_share": round(float((sub["alt_prev_n"] > 0).mean()), 4),
            "has_instalment_history_share": round(float((sub["alt_inst_n"] > 0).mean()), 4),
        }
    out["by_gender"] = df.groupby("audit_gender")["TARGET"].agg(["size", "mean"]).round(4).to_dict()
    out["by_age_band"] = df.groupby("audit_age_band")["TARGET"].agg(["size", "mean"]).round(4).to_dict()
    out["missing_share_top10"] = (df[bur_cols + alt_cols].isna().mean().sort_values(ascending=False)
                                  .head(10).round(4).to_dict())
    return out


# ---- Berka (PKDD'99) cash-flow features ----
CREDIT_TYPES = {"PRIJEM", "CREDIT"}
WITHDRAWAL_TYPES = {"VYDAJ", "VYBER", "WITHDRAWAL", "WITHDRAWAL IN CASH"}
SANCTION_SYMBOLS = {"SANKC. UROK", "SANCTION INTEREST IF NEGATIVE BALANCE"}


def _to_date(s):
    """Accept DATE values or YYMMDD integers from the original files."""
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    st = s.astype(str).str.replace(r"\.0$", "", regex=True)
    if st.str.fullmatch(r"\d{6}").all():
        return pd.to_datetime("19" + st, format="%Y%m%d", errors="coerce")
    return pd.to_datetime(st, errors="coerce")


def _gender_birth_from_number(bn):
    """Birth number YYMMDD; women have 50 added to the month."""
    st = bn.astype(str).str.zfill(6)
    mm = st.str[2:4].astype(int)
    gender = np.where(mm > 50, "F", "M")
    mm = np.where(mm > 50, mm - 50, mm)
    birth = pd.to_datetime("19" + st.str[:2] + pd.Series(mm, index=st.index).astype(str).str.zfill(2) + st.str[4:6],
                           format="%Y%m%d", errors="coerce")
    return pd.Series(gender, index=bn.index), birth


def build_berka(tables, window_days=365, min_history_days=90, gap_days=0):
    """tables: dict of DataFrames with keys loan, account, trans, disp, client. One row per loan."""
    loan = tables["loan"].copy(); acc = tables["account"].copy(); tr = tables["trans"].copy()
    disp = tables["disp"].copy(); cl = tables["client"].copy()
    for t in (loan, acc, tr, disp, cl):
        t.columns = [c.lower() for c in t.columns]
    loan["date"] = _to_date(loan["date"]); acc["date"] = _to_date(acc["date"]); tr["date"] = _to_date(tr["date"])
    tr["type"] = tr["type"].astype(str).str.upper().str.strip()
    tr["k_symbol"] = tr["k_symbol"].astype(str).str.upper().str.strip()

    loan["default"] = loan["status"].astype(str).str.upper().isin(["B", "D"]).astype(int)
    loan = loan.merge(acc[["account_id", "date"]].rename(columns={"date": "acct_open"}), on="account_id", how="left")

    owner = disp[disp["type"].astype(str).str.upper() == "OWNER"][["account_id", "client_id"]]
    if "birth_number" in cl.columns:
        cl["audit_gender"], cl["birth"] = _gender_birth_from_number(cl["birth_number"])
    else:  # pre-processed versions carry gender and birth_date directly
        cl["audit_gender"] = cl["gender"].astype(str).str.upper().str[0]
        cl["birth"] = pd.to_datetime(cl["birth_date"], errors="coerce")
    owner = owner.merge(cl[["client_id", "audit_gender", "birth"]], on="client_id", how="left").drop_duplicates("account_id")
    loan = loan.merge(owner[["account_id", "audit_gender", "birth"]], on="account_id", how="left")
    loan["audit_age"] = ((loan["date"] - loan["birth"]).dt.days / 365.25).round(1)

    tr = tr.merge(loan[["account_id", "date"]].rename(columns={"date": "loan_date"}), on="account_id")
    end = tr["loan_date"] - pd.Timedelta(days=gap_days)  # gap_days > 0 leaves a buffer before the loan date
    tr = tr[(tr["date"] < end) & (tr["date"] >= end - pd.Timedelta(days=window_days))]
    tr["is_credit"] = tr["type"].isin(CREDIT_TYPES)
    tr["inflow"] = np.where(tr["is_credit"], tr["amount"].abs(), 0.0)
    tr["outflow"] = np.where(~tr["is_credit"], tr["amount"].abs(), 0.0)
    tr["cash_out"] = np.where(tr["operation"].astype(str).str.upper().str.contains("VYBER|CASH"), tr["amount"].abs(), 0.0)
    tr["sanction"] = tr["k_symbol"].isin(SANCTION_SYMBOLS).astype(int)
    tr["month"] = tr["date"].dt.to_period("M")

    m = tr.groupby(["account_id", "month"]).agg(inflow=("inflow", "sum"), outflow=("outflow", "sum"),
                                                min_bal=("balance", "min")).reset_index()
    gm = m.groupby("account_id")
    feats = pd.DataFrame({
        "cf_months_observed": gm.size(),
        "cf_inflow_monthly_mean": gm["inflow"].mean(),
        "cf_inflow_cv": gm["inflow"].std() / gm["inflow"].mean().replace(0, np.nan),
        "cf_expense_to_income": gm["outflow"].sum() / gm["inflow"].sum().replace(0, np.nan),
        "cf_months_net_negative": (m.assign(neg=m["outflow"] > m["inflow"]).groupby("account_id")["neg"].mean()),
        "cf_min_balance": gm["min_bal"].min(),
    })
    gt = tr.groupby("account_id")
    feats["cf_balance_mean"] = gt["balance"].mean()
    feats["cf_balance_volatility"] = gt["balance"].std() / gt["balance"].mean().abs().replace(0, np.nan)
    feats["cf_low_balance_share"] = gt["balance"].apply(lambda b: float((b < 5000).mean()))
    feats["cf_negative_balance_share"] = gt["balance"].apply(lambda b: float((b < 0).mean()))
    feats["cf_sanction_count"] = gt["sanction"].sum()
    feats["cf_cash_withdrawal_share"] = gt["cash_out"].sum() / gt["outflow"].sum().replace(0, np.nan)
    feats["cf_txn_per_month"] = gt.size() / feats["cf_months_observed"]
    first = gt["date"].min()

    out = loan.merge(feats, left_on="account_id", right_index=True, how="left")
    out = out.merge(first.rename("first_txn"), left_on="account_id", right_index=True, how="left")
    out["cf_history_days"] = (out["date"] - out["first_txn"]).dt.days
    out["cf_account_tenure_years"] = (out["date"] - out["acct_open"]).dt.days / 365.25
    out["terms_amount"] = out["amount"]; out["terms_duration"] = out["duration"]; out["terms_payment"] = out["payments"]
    out["cf_payment_to_income"] = out["payments"] / out["cf_inflow_monthly_mean"].replace(0, np.nan)
    out["enough_history"] = (out["cf_history_days"] >= min_history_days).astype(int)
    keep = (["loan_id", "account_id", "date", "status", "default", "enough_history", "audit_gender", "audit_age"]
            + [c for c in out.columns if c.startswith("cf_") or c.startswith("terms_")])
    return out[keep].rename(columns={"date": "loan_date"})


def berka_summary(df):
    cf = [c for c in df.columns if c.startswith("cf_")]
    return {"n_loans": int(len(df)), "n_defaults": int(df["default"].sum()),
            "default_rate": round(float(df["default"].mean()), 4),
            "status_counts": df["status"].astype(str).value_counts().to_dict(),
            "loans_with_90d_history": int(df["enough_history"].sum()),
            "n_cashflow_features": len(cf),
            "by_gender": df.groupby("audit_gender")["default"].agg(["size", "mean"]).round(4).to_dict(),
            "cashflow_feature_means_by_outcome": df.groupby("default")[cf].mean().round(3).to_dict()}


# ---- Explainability, fairness and robustness (Notebook 3) ----
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

REASON_TEXT = {
    "bur_ext_source_1": "External credit score 1 is low", "bur_ext_source_2": "External credit score 2 is low",
    "bur_ext_source_3": "External credit score 3 is low", "bur_n_loans": "Number of credit bureau accounts",
    "bur_n_active": "Number of active credit accounts", "bur_oldest_years": "Length of credit history",
    "bur_newest_years": "Time since most recent credit account", "bur_max_days_overdue": "Days overdue on bureau accounts",
    "bur_total_credit": "Total credit extended", "bur_total_debt": "Total outstanding debt",
    "bur_total_overdue": "Amount currently overdue", "bur_n_prolonged": "Number of extended credit agreements",
    "bur_debt_to_credit": "Debt is high relative to credit", "bur_late_month_share": "Share of months with late payments",
    "alt_income": "Income level", "alt_credit_to_income": "Loan amount is high relative to income",
    "alt_annuity_to_income": "Repayment is high relative to income", "alt_credit_to_goods": "Loan amount relative to purchase price",
    "alt_employment_years": "Length of employment", "alt_not_employed": "Not currently employed",
    "alt_income_type": "Type of income", "alt_prev_n": "Number of previous applications",
    "alt_prev_approved_share": "Share of previous applications approved", "alt_prev_refused_share": "Share of previous applications refused",
    "alt_inst_n": "Number of past instalments", "alt_inst_late_share": "Share of past instalments paid late",
    "alt_inst_mean_days_late": "Average days late on past instalments", "alt_inst_max_days_late": "Longest delay on a past instalment",
    "alt_inst_short_share": "Share of past instalments underpaid", "alt_inst_paid_ratio": "Share of instalment amounts repaid",
    "alt_pos_dpd_share": "Past due on point-of-sale loans", "alt_pos_max_dpd": "Longest delay on point-of-sale loans",
    "alt_cc_util_mean": "Card balance is high relative to limit", "alt_cc_dpd_share": "Past due on card payments",
}


def lgb_contributions(model, X):
    """Exact TreeSHAP contributions from LightGBM (last column is the base value)."""
    return model.booster_.predict(X, pred_contrib=True)[:, :-1]


def top_reasons(contrib, cols, k=4):
    """Features pushing risk up the most, per applicant."""
    idx = np.argsort(-contrib, axis=1)[:, :k]
    return [[cols[j] for j in row if contrib[i, j] > 0] for i, row in enumerate(idx)]


def reason_stability(make_model_fn, X, y, cols, seeds, approve_rate=0.7, k=4, n_explain=5000):
    """Train one model per seed on the same split; compare adverse-action reasons for the same declined applicants."""
    Xtr, Xte, ytr, yte = train_test_split(X[cols], y, test_size=0.2, stratify=y, random_state=0)
    models = [make_model_fn(s).fit(Xtr, ytr) for s in seeds]
    p0 = models[0].predict_proba(Xte)[:, 1]
    declined = np.where(p0 > np.quantile(p0, approve_rate))[0][:n_explain]
    Xd = Xte.iloc[declined]
    reasons = [top_reasons(lgb_contributions(m, Xd), cols, k) for m in models]
    top1, jacc = [], []
    for a in range(len(seeds)):
        for b in range(a + 1, len(seeds)):
            for ra, rb in zip(reasons[a], reasons[b]):
                if ra and rb:
                    top1.append(ra[0] == rb[0])
                    sa, sb = set(ra), set(rb)
                    jacc.append(len(sa & sb) / len(sa | sb))
    return {"n_declined_explained": int(len(declined)), "top1_agreement": float(np.mean(top1)),
            "top4_jaccard_mean": float(np.mean(jacc))}, models[0], Xtr, Xte, yte, p0


def reason_faithfulness(model, Xtr, Xte, p_te, cols, approve_rate=0.7, n_explain=5000, seed=0):
    """Move each declined applicant's top reason to the typical approved value; compare with a random other feature."""
    rng = np.random.default_rng(seed)
    cut = np.quantile(p_te, approve_rate)
    dec = np.where(p_te > cut)[0][:n_explain]
    Xd = Xte.iloc[dec].copy()
    contrib = lgb_contributions(model, Xd)
    ref = Xtr.median(numeric_only=True)
    base = model.predict_proba(Xd)[:, 1]
    top = np.argmax(contrib, axis=1)
    Xt, Xr = Xd.copy(), Xd.copy()
    for i in range(len(Xd)):
        j = top[i]; Xt.iat[i, j] = ref[cols[j]]
        others = [c for c in range(len(cols)) if c != j]
        r = rng.choice(others); Xr.iat[i, r] = ref[cols[r]]
    pt, pr = model.predict_proba(Xt)[:, 1], model.predict_proba(Xr)[:, 1]
    return {"mean_pd_declined": float(base.mean()),
            "mean_drop_top_reason": float((base - pt).mean()), "mean_drop_random_feature": float((base - pr).mean()),
            "share_flipped_to_approve_top_reason": float((pt <= cut).mean()),
            "share_flipped_to_approve_random_feature": float((pr <= cut).mean())}


def global_importance(model, X, cols, n=20000, seed=0):
    Xs = X[cols].sample(min(n, len(X)), random_state=seed)
    c = np.abs(lgb_contributions(model, Xs)).mean(axis=0)
    return (pd.Series(c, index=cols).sort_values(ascending=False) / c.sum()).round(4)


def fairness_report(y, p, groups, approve_rate=0.7, reference=None):
    """Approval rates by group at one overall cutoff; four-fifths ratio and equal-opportunity gap for good borrowers."""
    y = np.asarray(y); p = np.asarray(p); g = pd.Series(np.asarray(groups))
    cut = np.quantile(p, approve_rate); ap = p <= cut
    rows = {}
    for k in sorted(g.dropna().unique()):
        m = (g == k).values
        rows[str(k)] = {"n": int(m.sum()), "approval_rate": float(ap[m].mean()),
                        "approval_rate_good_borrowers": float(ap[m & (y == 0)].mean()),
                        "approval_rate_defaulters": float(ap[m & (y == 1)].mean()),
                        "default_rate_among_approved": float(y[m & ap].mean())}
    ref = reference or max(rows, key=lambda k: rows[k]["approval_rate"])
    for k in rows:
        rows[k]["approval_ratio_vs_" + ref] = rows[k]["approval_rate"] / rows[ref]["approval_rate"]
        rows[k]["good_borrower_gap_vs_" + ref] = rows[k]["approval_rate_good_borrowers"] - rows[ref]["approval_rate_good_borrowers"]
    return rows

print('LightGBM available:', HAVE_LGB)

In [ ]:
# Load data and Notebook 2 predictions
p = f'{OUT}/home_credit_features.parquet'
hc = pd.read_parquet(p) if os.path.exists(p) else pd.read_csv(f'{OUT}/home_credit_features.csv.gz')
y_hc = hc['TARGET'].values
BUR = [c for c in hc.columns if c.startswith('bur_')]
ALT = [c for c in hc.columns if c.startswith('alt_')]
COMB = BUR + ALT
thin = hc['thin_file'].values == 1
oof = np.load(f'{OUT}/oof_predictions.npz')
print(list(oof.keys()))
results = {}

In [ ]:
# 1. Explainability: global drivers, reason stability across seeds, and faithfulness (about 10 minutes)
t0 = time.time()
stab, model0, Xtr, Xte, yte, p_te = reason_stability(lambda s: make_model('gbm', s), hc, y_hc, COMB, [11, 22, 33])
faith = reason_faithfulness(model0, Xtr, Xte, p_te, COMB)
imp = global_importance(model0, hc, COMB)
# most common adverse-action reasons for declined applicants, overall and thin-file
test_idx = Xte.index
cut = np.quantile(p_te, 0.7)
dec_mask = p_te > cut
reasons = top_reasons(lgb_contributions(model0, Xte[dec_mask]), COMB, k=4)
thin_dec = hc.loc[test_idx[dec_mask], 'thin_file'].values == 1
def tally(rs):
    s = pd.Series([REASON_TEXT.get(r, r) for row in rs for r in row]).value_counts(normalize=True)
    return s.head(10).round(4).to_dict()
results['explainability'] = {
    'stability': stab, 'faithfulness': faith,
    'global_importance_top15': imp.head(15).to_dict(),
    'share_of_importance_alternative': float(imp[[c for c in imp.index if c.startswith('alt_')]].sum()),
    'top_reasons_declined_all': tally(reasons),
    'top_reasons_declined_thin_file': tally([r for r, t in zip(reasons, thin_dec) if t])}
print(f'Done in {(time.time()-t0)/60:.1f} min')
print(json.dumps(results['explainability'], indent=1)[:3000])

In [ ]:
# 2. Fairness by gender and age band (uses Notebook 2 out-of-fold predictions)
fair = {}
for model_name in ['gbm_bureau', 'gbm_combined']:
    p = oof[f'exp1_{model_name}']
    fair[model_name] = {
        'gender_all': fairness_report(y_hc, p, hc['audit_gender'], reference='M'),
        'age_all': fairness_report(y_hc, p, hc['audit_age_band'], reference='30-49'),
        'gender_thin_file': fairness_report(y_hc[thin], p[thin], hc.loc[thin, 'audit_gender'], reference='M'),
        'age_thin_file': fairness_report(y_hc[thin], p[thin], hc.loc[thin, 'audit_age_band'], reference='30-49')}
results['fairness'] = fair
for k, v in fair.items():
    print('\n==', k)
    for g, rows in v.items():
        print(g, {grp: (round(r['approval_rate'], 3), round(r['approval_rate_good_borrowers'], 3)) for grp, r in rows.items()})

In [ ]:
# 3. Berka robustness (about 5 minutes)
BK = f'{OUT}/berka_raw'
tables = {n: pd.read_csv(f'{BK}/{n}.csv') for n in ['loan', 'account', 'trans', 'disp', 'client']}
def run_berka(bk, cols, label, seeds=list(range(20))):
    y = bk['default'].values
    out = {}
    for kind in ['logit', 'gbm']:
        P = oof_predictions(bk[cols], y, kind, seeds, small_data=True)
        r = evaluate(y, P, None, 1000)
        out[kind] = {k: r[k] for k in ['n', 'default_rate', 'auc', 'auc_ci95', 'pr_auc', 'brier']}
    print(label, {k: (round(v['auc'], 3), [round(x, 3) for x in v['auc_ci95']]) for k, v in out.items()})
    return out
base = build_berka(tables)
CF = [c for c in base.columns if c.startswith('cf_')]
TERMS = [c for c in base.columns if c.startswith('terms_')]
rob = {}
rob['base_cashflow'] = run_berka(base, CF, 'base')
rob['without_sanction_and_negative_balance'] = run_berka(
    base, [c for c in CF if c not in ('cf_sanction_count', 'cf_negative_balance_share', 'cf_min_balance')], 'no sanction/neg/min')
gap = build_berka(tables, gap_days=30)
rob['gap_30_days_before_loan'] = run_berka(gap, CF, 'gap 30d')
fin = base[base['status'].astype(str).isin(['A', 'B'])].reset_index(drop=True)
rob['finished_loans_only'] = run_berka(fin, CF, f'finished only ({len(fin)} loans, {int(fin.default.sum())} bad)')
rob['terms_only_reference'] = run_berka(base, TERMS, 'terms only')
results['berka_robustness'] = rob
results['berka_fairness_gender'] = fairness_report(base['default'].values, oof['exp3_gbm_cashflow'],
                                                   base['audit_gender'], reference='M')

In [ ]:
# Save and download
with open(f'{OUT}/explain_results.json', 'w') as f:
    json.dump(results, f, indent=1, default=str)
from google.colab import files
files.download(f'{OUT}/explain_results.json')
print('Saved. Send explain_results.json to Claude.')